<a href="https://colab.research.google.com/github/FahimAlaviRidoy/Assisment2MLDL/blob/main/Assessment_2_(NLP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
pip install pymupdf sentence-transformers faiss-cpu transformers torch

In [17]:
import fitz  # PyMuPDF
import re
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [18]:
# ---------------------------
# 1. Load and preprocess PDF
# ---------------------------
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [19]:
def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,!? ]', '', text)
    return text.strip()

In [20]:
# ---------------------------
# 2. Chunking
# ---------------------------
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

# ---------------------------
# 3. Embedding (Feature Engineering)
# ---------------------------
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def create_embeddings(chunks):
    embeddings = embedder.encode(chunks)
    return embeddings

# ---------------------------
# 4. Build FAISS index
# ---------------------------
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
# ---------------------------
# 5. Load QA model (local)
# ---------------------------
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2"
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
# ---------------------------
# 6. Retrieve + Answer
# ---------------------------
def answer_question(question, chunks, index, k=3):
    query_embedding = embedder.encode([question])
    distances, indices = index.search(query_embedding, k)

    # Combine top chunks
    context = " ".join([chunks[i] for i in indices[0]])

    result = qa_pipeline(question=question, context=context)

    return result['answer']

# ---------------------------
# 7. Main Pipeline
# ---------------------------
def build_qa_system(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    clean = clean_text(raw_text)
    chunks = chunk_text(clean)

    embeddings = create_embeddings(chunks)
    index = build_faiss_index(embeddings)

    return chunks, index


In [23]:
# ---------------------------
# Example Usage
# ---------------------------
if __name__ == "__main__":
    pdf_path = "/content/Test PDF (Hard).pdf"

    print("Processing PDF...")
    chunks, index = build_qa_system(pdf_path)

    while True:
        question = input("\nAsk a question (or 'exit'): ")
        if question.lower() == 'exit':
            break

        answer = answer_question(question, chunks, index)
        print("Answer:", answer)

Processing PDF...

Ask a question (or 'exit'): What is the capital city of Bangladesh?  
Answer: Dhaka

Ask a question (or 'exit'): Which river is considered the lifeline of Bangladesh? 
Answer: Padma River

Ask a question (or 'exit'): In which year did Bangladesh gain independence? 
Answer: 1971

Ask a question (or 'exit'): What is the official language of Bangladesh? 
Answer: Bengali

Ask a question (or 'exit'): What is the name of the world’s largest mangrove forest located in Bangladesh?  
Answer: Sundarbans

Ask a question (or 'exit'): Which currency is used in Bangladesh? What is the national animal of Bangladesh?  
Answer: Royal Bengal Tiger

Ask a question (or 'exit'): Which currency is used in Bangladesh? What is the national animal of Bangladesh?  
Answer: Royal Bengal Tiger

Ask a question (or 'exit'): Which currency is used in Bangladesh?
Answer: Bangladeshi Taka

Ask a question (or 'exit'): exit
